In [17]:
import os
HOME = "/Users/chandler/Downloads/CVCP"  # Set your project directory path
import ultralytics
from ultralytics import YOLO # Import YOLO class. This class is used to create a YOLOv8 model
from IPython.display import display, Image
from roboflow import Roboflow
import torch
from tqdm import tqdm
from ultralytics.nn.tasks import DetectionModel
import torch.serialization
from torch.nn.modules.container import Sequential

print(f"Project directory: {HOME}")
HOME

Project directory: /Users/chandler/Downloads/CVCP


'/Users/chandler/Downloads/CVCP'

_______________________________________________________________________________________________

_______________________________________________________________________________________________

# Training the model
modify the /data_path/ yourselve

modify data.yaml file as well

In [ ]:
%cd {HOME}
HOME
data_path= "/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml"
model = YOLO("yolov8n.yaml")
results = model.train(data= data_path, epochs=50, imgsz=640, plots=True)

#/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml

#Model Fine-Tuning


In [20]:
# Fine-tune YOLOv8n on the weed/crop dataset
%cd {HOME}

# Resolve the dataset root so we can point explicitly to the datasets_yolocp/weed-crop-aerial directories
candidate_roots = [
    f"{HOME}/datasets_yolocp/weed-crop-aerial",
    f"{HOME}/datasets_yolocp/weed-crop-aerial-2",
]
dataset_root = next((path for path in candidate_roots if os.path.isdir(path)), None)
if dataset_root is None:
    raise FileNotFoundError("Could not find datasets_yolocp/weed-crop-aerial[(-2)] under HOME.")

tune_split = f"{dataset_root}/valid"
val_split = f"{dataset_root}/valid"
test_split = f"{dataset_root}/test"

weed_crop_data = {
    "path": dataset_root,
    "train": tune_split,
    "val": val_split,
    "test": test_split,
    "names": ["crop", "weed"],
    "nc": 2,
}

print(f"Tuning on validation split: {weed_crop_data['train']}")

candidate_weight_paths = [
    f"{HOME}/(example)runs/detect/weights/best.pt",
    f"{HOME}/(example)runs/detect/train/weights/best.pt",
]
weight_path = next((path for path in candidate_weight_paths if os.path.isfile(path)), None)
if weight_path is None:
    raise FileNotFoundError("Could not find (example)runs/detect[/train]/weights/best.pt under HOME.")

print(f"Using pretrained weights from: {weight_path}")



import ultralytics.nn.modules as ul_modules
safe_classes = {DetectionModel, Sequential}
for attr in dir(ul_modules):
    obj = getattr(ul_modules, attr)
    if isinstance(obj, type):
        safe_classes.add(obj)

torch.serialization.add_safe_globals(list(safe_classes))

# Monkey-patch torch.load to always use weights_only=False for YOLO checkpoint loading
orig_torch_load = torch.load
def torch_load_hacked(*args, **kwargs):
    kwargs['weights_only'] = False
    return orig_torch_load(*args, **kwargs)
torch.load = torch_load_hacked

tune_model = YOLO(weight_path)

torch.load = orig_torch_load  # restore the original function immediately after

tuning_args = dict(
    data=weed_crop_data,
    epochs=75,
    batch=16,
    imgsz=640,
    lr0=0.0025,
    lrf=0.05,
    optimizer="SGD",
    warmup_epochs=5,
    patience=50,
    close_mosaic=5,
    mosaic=0.0,  # disabled to avoid augmentation-related errors
    mixup=0.0,   # disabled to avoid augmentation-related errors
    cos_lr=True,
    dropout=0.05,
    weight_decay=5e-4,
    project=f"{HOME}/runs",
    name="tune_yolov8n_valid",
    verbose=True,
)

import yaml
# Save dynamically generated dict to yaml file
import os
tune_yaml_path = os.path.join(HOME, "runs/tune_yolov8n_valid/tune_data.yaml")
os.makedirs(os.path.dirname(tune_yaml_path), exist_ok=True)
with open(tune_yaml_path, 'w') as f:
    yaml.dump(weed_crop_data, f)

# Now point to the file, not the dict
tuning_args['data'] = tune_yaml_path

# Launch fine-tuning run
tune_results = tune_model.train(**tuning_args)
tune_results


This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.


/Users/chandler/Downloads/CVCP
Tuning on validation split: /Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/valid
Using pretrained weights from: /Users/chandler/Downloads/CVCP/(example)runs/detect/train/weights/best.pt
New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.27 🚀 Python-3.10.18 torch-2.8.0 CPU (Apple M4)
engine/trainer: task=detect, mode=train, model=/Users/chandler/Downloads/CVCP/(example)runs/detect/train/weights/best.pt, data=/Users/chandler/Downloads/CVCP/runs/tune_yolov8n_valid/tune_data.yaml, epochs=75, time=None, patience=50, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=/Users/chandler/Downloads/CVCP/runs, name=tune_yolov8n_valid3, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=5, resume=False, amp=True, fraction=1.0, profile=False, fre

train: Scanning /Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/valid/labels.cache... 235 images, 0 backgrounds, 0 corrupt: 100%|██████████| 235/235 [00:00<?, ?it/s]
'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
val: Scanning /Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/valid/labels.cache... 235 images, 0 backgrounds, 0 corrupt: 100%|██████████| 235/235 [00:00<?, ?it/s]

Plotting labels to /Users/chandler/Downloads/CVCP/runs/tune_yolov8n_valid3/labels.jpg... 


optimizer: SGD(lr=0.0025, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/chandler/Downloads/CVCP/runs/tune_yolov8n_valid3
Starting training for 75 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


  0%|          | 0/15 [00:00<?, ?it/s]


error: OpenCV(4.9.0) :-1: error: (-5:Bad argument) in function 'warpAffine'
> Overload resolution failed:
>  - M is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'M'


In [22]:
import os
label_dir = '/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/valid/labels'
img_dir = '/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/valid/images'
bad_labels = []
missing_labels = []
unexpected = []

# Collect image IDs (w/o extension for match-up)
img_ids = {os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
label_ids = {os.path.splitext(f)[0] for f in os.listdir(label_dir) if f.lower().endswith('.txt')}

for img_id in img_ids:
    label_path = os.path.join(label_dir, img_id + '.txt')
    if not os.path.exists(label_path):
        missing_labels.append(img_id)
    else:
        with open(label_path) as f:
            lines = [line.strip() for line in f]
            if not lines or all(l == "" for l in lines):
                bad_labels.append(label_path)
            for line in lines:
                parts = line.split()
                if len(parts) < 5:
                    unexpected.append((label_path, line, "Too few fields"))
                else:
                    cls, x, y, w, h = parts[:5]
                    try:
                        if int(cls) not in [0, 1]:  # Only two classes expected
                            unexpected.append((label_path, line, "Unexpected class"))
                        for v in [x, y, w, h]:
                            fval = float(v)
                            if not (0 <= fval <= 1):
                                unexpected.append((label_path, line, "Value out of bounds"))
                    except Exception as e:
                        unexpected.append((label_path, line, f"Parse error: {e}"))

print("Missing label files:", missing_labels)
print("Labels missing or empty:", bad_labels)
print("Other unexpected label issues:", unexpected)


Missing label files: []
Labels missing or empty: []
Other unexpected label issues: []


_______________________________________________________________________________________________

# Model Evaluation
When we are analysing how well YOLO is at predicting the contents of an image, there are several metrics we can use.
The most important ones are the **training loss** and the **validation loss**. The lower these values are, the better your algorithm is at predicting data. 

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

# Furthermore, here is the F-1 Curve
The F-1 curve tells us the overall performance of our model. It is particularly insightful because it **accounts for underrepresented classes**.
Imagine you have a thousand pictures of dogs and five of cats. You might have high accuracy if you always output dogs, but your F1 score will reflect this issue. 

In [ ]:
Image(filename=f'{HOME}/runs/detect/train/F1_curve.png', width=600)

_______________________________________________________________________________________________

## Testing the model
Previously, the model only saw pictures in the **train** folder. Now, we will show it the pictures in the **test** folder, pictures the model has never seen before. Based on how good the model's performance is with the test images, we can have an idea of what the model's performance with data in the real world will be.

## Test our model

In [ ]:
# Load a model
%cd {HOME}
model_path=f"{HOME}/runs/detect/train/weights/best.pt"
model_2 = YOLO(model_path)  # our trained YOLOv8n model

# Run batched inference on a list of images
results_2 = model_2(test1) 

# Process results list
for result in results_2:
    result.show()  # display to screen